# Single-Lane Implementation

We implemented the Nagel-Schreckenberg in a single-lane.

In [1]:
import numpy as np

class Traffic:
    """
    A Nagel-Schreckenberg traffic cellular automaton simulation.

    The road is represented by a 1D array , where each cell is either empty (-1)
    or occupied by a car with a certain velocity (integer > 0).

    Parameters:
    n (int): Length of the road (number of cells)
    density (float): Fraction of cells initially containing cars (0 to 1)
    vmax (int): Maximum allowed velocity
    p (float): Random slowdown probability (0 to 1)
    grid (np.ndarray): The grid of car velocities or -1 for empty
    history (list): A list of configurations of the grid at each step
    """

    def __init__(self, n=100, density=0.3, vmax=5, p=0.2, random_state=None):
        np.random.seed(random_state)

        self.n = n
        self.vmax = vmax
        self.p = p
        
        # Random initial placement of cars
        grid = -1 * np.ones(n, dtype=int)
        car_positions = np.random.choice(np.arange(n), size=int(density*n), replace=False)
        for pos in car_positions:
            grid[pos] = np.random.randint(1, vmax+1)  

        self.grid = grid
        self.history = [self.grid.copy()] 

    def step(self):
        """
        Performs a single step of the Nagel-Schreckenberg model.
        Each step consists of four actions applied in parallel:
        1. Acceleration
        2. Slowing down due to cars ahead
        3. Randomization (slow down with probability p)
        4. Car motion (cars move forward by their velocities)
        """
        grid = self.grid.copy()
        new_grid = -1 * np.ones_like(grid)

        # 1. Acceleration
        velocities = np.where(grid >= 0, np.minimum(grid + 1, self.vmax), -1)

        # 2. Slowing down
        for i in range(self.n):
            if velocities[i] >= 0: 
                distance = 1
                while grid[(i + distance) % self.n] == -1 and distance <= self.vmax:
                    distance += 1
                distance -= 1  
                velocities[i] = min(velocities[i], distance)

        # 3. Randomization
        rand = np.random.rand(self.n)
        velocities = np.where((velocities > 0) & (rand < self.p), velocities - 1, velocities)

        # 4. Car motion 
        for i in range(self.n):
            v = velocities[i]
            if v >= 0:
                new_pos = (i + v) % self.n
                new_grid[new_pos] = v

        self.grid = new_grid
        self.history.append(self.grid.copy())
        return self.grid
    
    def average_flow(self):
        """
        Compute the average flow: 
        flow = density * average velocity = (#cars / n) * mean velocity of cars
        """
        cars = self.grid[self.grid >= 0]          
        if len(cars) == 0:
            return 0
        return (len(cars) / self.n) * np.mean(cars)


    def jam_fraction(self):
        """
        Fraction of cars that are part of a jam.
        A 'jam' is defined as a car with velocity 0.
        """
        cars = self.grid[self.grid >= 0]
        if len(cars) == 0:
            return 0
        return np.sum(cars == 0) / len(cars)


    def simulate(self, n_step):
        """Run the automaton for n_step iterations."""
        for _ in range(n_step):
            self.step()
        return self.grid


# Multi-Lane Implementation

Here we implemented a 2D array, with the rows of our system as single-lanes for the Nagel-Schreckenberg model, and then implemented our own multi-lane rules.

In [206]:
class Multi_Lane_Traffic(Traffic):
    """
    A Nagel-Schreckenberg traffic multi-lane cellular automaton simulation.

    The road is represented by a 2D array , where each cell is either empty (-1)
    or occupied by a car with a certain velocity (integer > 0).
    

    Parameters:
    cells (int): Number of cells (cols)
    lanes (int): Number of lanes (rows)
    density (float): Fraction of cells initially containing cars (0 to 1)
    vmax (int): Maximum allowed velocity
    p (float): Random slowdown probability (0 to 1)
    grid (np.ndarray): The grid of car velocities or -1 for empty
    history (list): A list of configurations of the grid at each step
    """
    def __init__(self, cells=100, lanes=None, density=0.3, vmax=5, p=0.2, random_state=None, tolerance_limit = None):
        self.random_state = random_state    # Sets the random state of our simulation.
        np.random.seed(self.random_state)
        self.tolerance_limit = tolerance_limit

        if lanes == None:           # The number of rows.
            self.lanes = cells
        else:
            self.lanes = lanes

        self.cells = cells          # The number of cols.
        self.vmax = vmax            # The max velocity of the system.
        self.p = p                  # The probability that our velocity will decrease.
        
        
        # Random initial placement of cars across the rows and lanes.
        grid = []
        #print(grid)
        for lane in range(self.lanes):
            grid.append( Traffic(self.cells, density, self.vmax, self.p).grid )

        #print(grid)
        

        
        self.grid = (np.array(grid))

        
        self.history = [self.grid.copy()] 


    def step(self):
        """
        Performs a single step of the Nagel-Schreckenberg model for cells that don't have to switch lanes.
        Applies the lane switching rules with the Nagel-Schreckenberg model for all other cells.
        The lane switching rules are as follows:
            if velocity > distance:
                move to the lane which gives the max travel distance
                if the max travel distances are equal
                choose to move left or right at random
        Each lane-change cell will be simulated sequentially at a random order.

        """
        np.random.seed(self.random_state)

        lane_switch_cells = []
        grid = self.grid.copy()
        new_grid = -1 * np.ones_like(grid)

        # 1. Acceleration
        velocities = np.where(grid >= 0, np.minimum(grid + 1, self.vmax), -1)
        

        # 2. Slowing down
        for lane in range(self.lanes):
            for cell in range(self.cells):
                bumping_flag = False
                vel = velocities[lane][cell]
                if velocities[lane][cell] >= 0: 
                    distance = 1
                    bumping_flag = ( grid[lane][(cell + distance) % self.cells] == -1 )
                    while bumping_flag and distance <= self.vmax:
                        distance += 1
                        bumping_flag = ( grid[lane][(cell + distance) % self.cells] == -1 )
                    distance -= 1
                    if bumping_flag and not (grid[(lane-1) % self.lanes][cell] == -1 and grid[(lane+1) % self.lanes][cell] == -1):
                        lane_switch_cells.append((lane,cell))
                    else:
                        velocities[lane][cell] = min(velocities[lane][cell], distance)



        # 3. Randomization
        rand = np.random.rand(self.cells)
        velocities = np.where((velocities > 0) & (rand < self.p), velocities - 1, velocities)

        #lane_switch_cells = np.random.shuffle(lane_switch_cells)

        # 4. Car motion 
        for lane in range(self.lanes):             # Updates the cars that go in a linear motion
            for cell in range(self.cells):
                if [lane,cell] in lane_switch_cells:
                    continue
                vel = velocities[lane][cell]
                if vel >= 0:
                    new_pos = [lane,(cell + vel) % self.cells]
                    new_grid[new_pos[0]][new_pos[1]] = vel
                
        for pos in lane_switch_cells:       # Updates the cars that switched lanes
            lane, cell = pos[0], pos[1]
            left_pos, right_pos = [(lane-1) % self.lanes, cell], [(lane+1) % self.lanes, cell]  # Defines the left and right positions.
            vel = velocities[lane][cell]          # This gets the velocity of the lane we are switching to.
            left_distance, right_distance = 0, 0
            if vel >= 0:

                if grid[left_pos[0]][ left_pos[1]] != -1:       # Finds the max distance for moving in the left lane.
                    left_distance = 1
                    bumping_flag = ( grid[left_pos[0]][(cell + left_distance) % self.cells] == -1 )
                    while bumping_flag and left_distance <= self.vmax:
                        left_distance += 1
                        bumping_flag = ( grid[left_pos[0]][(cell + left_distance) % self.cells] == -1 )
                    left_distance -= 1
                    left_distance = min(velocities[left_pos[0]][left_pos[1]], left_distance)

                if grid[right_pos[0]][right_pos[1]] != -1:       # Finds the max distance for moving in the right lane.
                    right_distance = 1
                    bumping_flag = ( grid[right_pos[0]][(cell + right_distance) % self.cells] == -1 )
                    while bumping_flag and right_distance <= self.vmax:
                        right_distance += 1
                        bumping_flag = ( grid[right_pos[0]][(cell + right_distance) % self.cells] == -1 )
                    right_distance -= 1
                    right_distance = min(velocities[right_pos[0]][right_pos[1]], left_distance)

                    # Finds the max distance for moving in the center lane.
                distance = 1
                bumping_flag = ( grid[pos[0]][(cell + distance) % self.cells] == -1 )
                while bumping_flag and distance <= self.vmax:
                    distance += 1
                    bumping_flag = ( grid[pos[0]][(cell + distance) % self.cells] == -1 )
                distance -= 1
                distance = min(velocities[pos[0]][pos[1]], left_distance)


            # Chooses the farthest distance and update the grid accordingly.
            if left_distance > right_distance and left_distance > distance:
                new_pos = [left_distance, left_pos[1]]

            elif right_distance > left_distance and right_distance > distance:
                new_pos = [right_distance, right_pos[1]]

            elif distance > left_distance and distance > right_distance:
                new_pos = [distance, lane]

            elif left_distance == right_distance and left_distance > distance:
                choise = np.random.choice(["left","right"])
                if choise == "left":
                    new_pos = [left_distance, left_pos[1]]
                else:
                    new_pos = [right_distance, right_pos[1]]
                
            elif left_distance == right_distance and left_distance == distance:
                choise = np.random.choice(["left","right","center"])
                if choise == "left":
                    new_pos = [left_distance, left_pos[1]]
                elif choise == "right":
                    new_pos = [right_distance, right_pos[1]]
                else:
                    new_pos = [distance, lane]
            elif left_distance == distance:
                choise = np.random.choice(["left","center"])
                if choise == "left":
                    new_pos = [left_distance, left_pos[1]]
                else:
                    new_pos = [distance, cell]
            elif right_distance == distance:
                choise =np.random.choice(["right","center"])
                if choise == "right":
                    new_pos = [right_distance, right_pos[1]]
                else:
                    new_pos = [distance, cell]

            else:
                    new_pos = [distance, cell]

            new_grid[new_pos[0] % self.lanes][new_pos[1] % self.cells] = vel
                    
                
            

            
            



        self.grid = new_grid
        self.history.append(self.grid.copy())
        return self.grid
    
    def average_flow(self, grid):
        """
        Compute the average flow: 
        flow = density * average velocity = (#cars / n) * mean velocity of cars
        """
        cars = grid[grid >= 0]          
        if len(cars) == 0:
            return 0
        return (len(cars) / (self.cells * self.lanes)) * np.mean(cars)
    
    def history_average_flow(self):
        """
        The average "average flow" over the total simulation history.
        """
        sum_average_flow, counter = 0, 0
        for grid in self.history:
            counter += 1
            sum_average_flow += self.average_flow(grid)
        return sum_average_flow / counter


    def jam_fraction(self, grid):
        """
        Fraction of cars that are part of a jam.
        A 'jam' is defined as a car with velocity 0.
        """
        cars = grid[grid >= 0]
        if len(cars) == 0:
            return 0
        return np.sum(cars == 0) / len(cars)
    
    def history_jam_fraction(self):
        """
        The average "jam fraction" over the total simulation history.
        """
        sum_jam_fraction, counter = 0, 0
        for grid in self.history:
            counter += 1
            sum_jam_fraction += self.jam_fraction(grid)
        
        return sum_jam_fraction / counter


    def simulate(self, n_step):
        """Run the automaton for n_step iterations."""
        
        Previous_Stats = np.array([self.history_average_flow(), self.history_jam_fraction()])

        for _ in range(n_step):
            self.step()

            # This allows the simulation to stop if the statistical measurements reach a steady state, i.e. the difference between the previous and current state is less than the tolerance.
            if self.tolerance_limit != None:
                Current_Stats = np.array([self.history_average_flow(), self.history_jam_fraction()])
                if np.linalg.norm( Previous_Stats - Current_Stats ) < self.tolerance_limit:
                    break
                Previous_Stats = Current_Stats

            

        return self.grid
        




# Testing

## Single Lane Testing

This system tests the spawning of our single-lane implementation, as well as the metrics we used for analysis.

In [189]:
model = Traffic(
    n=100,        
    density=0.3,  
    vmax=5,       
    p=0.3,        
    random_state=42
)

model.simulate(0)


array([ 1, -1, -1, -1,  5, -1, -1, -1, -1,  1,  2, -1,  3, -1, -1,  5, -1,
       -1,  4, -1, -1, -1,  5, -1, -1, -1,  5, -1, -1, -1,  1,  3, -1,  2,
       -1, -1, -1, -1, -1,  3,  2, -1,  3, -1,  1,  3, -1, -1, -1, -1, -1,
       -1, -1,  1, -1,  1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1,  1,  3, -1,  2,  4, -1, -1,  3,  4, -1, -1,  2, -1, -1,  3, -1,
       -1, -1, -1,  3, -1,  1, -1, -1, -1, -1, -1,  3, -1, -1, -1])

In [193]:
# Test 1: empty road -> flow = jam = 0
model = Traffic(n=20, density=0.5, vmax=5, p=0)
model.simulate(0)

flow = model.average_flow()
jam = model.jam_fraction()

print("Test 1: Empty Road")
print(f"  Expected flow = 0, Actual flow = {flow}")
print(f"  Expected jam fraction = 0, Actual jam fraction = {jam}")
print("  PASS" if flow == 0 and jam == 0 else "  FAIL")
print()


# Test 2: full road → most cars must jam jam = 1
model = Traffic(n=20, density=1, vmax=5, p=0)
model.simulate(10)

jam = model.jam_fraction()

print("Test 2: Full Road")
print(f"  Expected jam fraction = 1, Actual jam fraction = {jam:.3f}")
print("  PASS" if jam > 0.9 else "  FAIL")





Test 1: Empty Road
  Expected flow = 0, Actual flow = 1.6
  Expected jam fraction = 0, Actual jam fraction = 0.0
  FAIL

Test 2: Full Road
  Expected jam fraction = 1, Actual jam fraction = 1.000
  PASS


## Mutli-Lane Testing

This tested the spawning of a multi-lane model, as well as the simulation and our metric functions.

In [356]:
Model = Multi_Lane_Traffic(
                        cells=1000,
                        lanes=1,
                        density=0.01,
                        vmax=5,
                        p=0,
                        random_state=10,
                        )

Model.simulate(100)

print("Totoal Average Flow:", Model.history_average_flow())
print("Total Jamming Fraction:", Model.history_jam_fraction())
print(np.matrix_transpose(Model.grid))

Totoal Average Flow: 0.5453564356435646
Total Jamming Fraction: 0.09762144481297551
[[-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [ 0]
 [-1]
 [ 1]
 [ 0]
 [-1]
 [ 1]
 [-1]
 [ 1]
 [-1]
 [ 1]
 [-1]
 [-1]
 [ 2]
 [-1]
 [ 1]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [-1]
 [-1]
 [ 2]
 [ 0]
 [-1]
 [ 1]
 [ 0]
 [-1]
 [ 1